# Data preparation

## MIPD

Builds three files in `data/mipd/`:

* `articles.csv` - `article_id`, `full_article`, `category` (`raw_articles.csv` joined with `articles_categories.csv`).
* `intention.csv` - `article_id`, `original_set`, intention labels (train + validation + test unioned).
* `manipulation.csv` - `article_id`, `original_set`, manipulation labels (train + validation + test unioned).

The split files have no id, so they are linked to `raw_articles.csv` by exact article text (verified 1:1).

Run from the repository root or from `notebooks/`.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
MIPD = ROOT / "data" / "mipd"
SPLITS = ["train", "validation", "test"]

In [2]:
raw = pd.read_csv(MIPD / "raw_articles.csv")
categories = pd.read_csv(MIPD / "articles_categories.csv")

articles = raw.merge(categories, on="article_id", how="inner", validate="one_to_one")
assert len(articles) == len(raw) == len(categories)
assert articles["full_article"].is_unique

articles.to_csv(MIPD / "articles.csv", index=False)
print(articles.shape)
articles["category"].value_counts()

(15356, 3)


category
COVID                        6049
WOMENS_RIGHTS                1064
FIVE_G                       1048
ASTRONOMY_AND_AERONAUTICS    1046
LGBT                         1036
NEWS_OR_OTHER                1033
MIGRATIONS                   1030
THE_WAR_IN_UKRAINE           1026
PSEUDOMEDICINE               1013
CLIMATE_CRISIS               1011
Name: count, dtype: int64

In [3]:
text_to_id = articles.set_index("full_article")["article_id"]


def build_task(task: str) -> pd.DataFrame:
    df = pd.concat(
        [pd.read_csv(MIPD / task / f"{s}.csv").assign(original_set=s) for s in SPLITS],
        ignore_index=True,
    )
    df["article_id"] = df["article"].map(text_to_id)
    assert df["article_id"].notna().all(), f"{task}: unmatched articles"
    assert df["article_id"].is_unique, f"{task}: article in more than one split"

    labels = [c for c in df.columns if c not in ("article", "original_set", "article_id")]
    df["article_id"] = df["article_id"].astype(int)
    df[labels] = df[labels].astype(int)
    return df[["article_id", "original_set", *labels]].sort_values("article_id", ignore_index=True)


tasks = {task: build_task(task) for task in ["intention", "manipulation"]}
for task, df in tasks.items():
    assert set(df["article_id"]) == set(articles["article_id"])
    df.to_csv(MIPD / f"{task}.csv", index=False)
    print(task, df.shape)
    display(df["original_set"].value_counts())

intention (15356, 11)


original_set
train         10749
validation     3086
test           1521
Name: count, dtype: int64

manipulation (15356, 13)


original_set
train         10749
validation     3086
test           1521
Name: count, dtype: int64

In [4]:
# Sanity check: reload and join back to articles
intention = pd.read_csv(MIPD / "intention.csv")
manipulation = pd.read_csv(MIPD / "manipulation.csv")
assert (intention[["article_id", "original_set"]] == manipulation[["article_id", "original_set"]]).all().all()

intention.merge(articles, on="article_id").head()

,article_id,original_set,NEGATING_SCIENTIFIC_FACTS,UNDERMINING_THE_CREDIBILITY_OF_PUBLIC_INSTITUTIONS,CHALLENGING_AN_INTERNATIONAL_ORGANISATION,PROMOTING_STEREOTYPES,WEAKENING_INTERNATIONAL_ALLIANCES,UNDERMINING_INTERNATIONAL_COUNTRY_POSITION,CHANGE_OF_ELECTION_BELIEFS,RAISING_MORALE_OF_ONE_SIDE_OF_CONFLICT,CAUSE_PANIC,full_article,category
0,0,test,0,0,0,0,0,0,0,0,0,"""Co przyczyną zwiększonej liczby zakażeń? Wice...",COVID
1,1,train,1,0,0,0,0,0,0,0,0,"""Wynalazca technologii mRNA apeluje: nie szcze...",COVID
2,2,train,0,0,0,0,0,0,0,0,0,"""Tajwan przygotowuje schrony. To odpowiedź na ...",NEWS_OR_OTHER
3,3,validation,1,1,1,1,0,0,0,0,0,"""Paszport covidowy to koń trojański systemu kr...",COVID
4,4,train,0,0,1,0,1,0,0,0,0,"""Migranci są bronią Putina: w Polsce nagle ujr...",MIGRATIONS


## SemEval-2023 Task 3

Creates `data/sem_eval/en.csv` and `data/sem_eval/pl.csv` by joining paragraph text and labels from the original train and dev sets. Each row contains `article_id`, `paragraph_number`, `original_set`, `text`, and one binary column per technique in `task3_def.csv`. A paragraph without techniques has zeros in all label columns.

Some Polish training template rows are concatenated without a newline. Their boundaries are recovered using the missing article and paragraph identifiers from the labels file; the join is checked for missing or duplicate rows.

In [1]:
from pathlib import Path
import re

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
SEM_EVAL = ROOT / "data" / "sem_eval"
TECHNIQUES = pd.read_csv(SEM_EVAL / "task3_def.csv", header=None)[0].tolist()
KEYS = ["article_id", "paragraph_number"]


def load_semeval_split(language: str, original_set: str) -> pd.DataFrame:
    base = SEM_EVAL / language / f"{original_set}-labels-subtask-3"
    with base.with_suffix(".txt").open(encoding="utf-8") as source:
        label_rows = [line.rstrip("\r\n").split("\t", 2) for line in source if line.strip()]
    label_rows = [row + [""] * (3 - len(row)) for row in label_rows]
    labels = pd.DataFrame(label_rows, columns=[*KEYS, "labels"])
    if labels.duplicated(KEYS).any():
        raise ValueError(f"{language}/{original_set}: duplicate label keys")

    with base.with_suffix(".template").open(encoding="utf-8") as source:
        text_rows = [line.rstrip("\r\n").split("\t", 2) for line in source if line.strip()]
    if any(len(row) != 3 for row in text_rows):
        raise ValueError(f"{language}/{original_set}: malformed template row")

    missing_keys = set(map(tuple, labels[KEYS].to_numpy())) - {tuple(row[:2]) for row in text_rows}
    if missing_keys:
        boundary = re.compile("|".join(re.escape(f"{article_id}\t{paragraph_number}\t") for article_id, paragraph_number in sorted(missing_keys)))
        recovered_rows = []
        for article_id, paragraph_number, text in text_rows:
            matches = list(boundary.finditer(text))
            previous_end = 0
            current_key = (article_id, paragraph_number)
            for match in matches:
                recovered_rows.append([*current_key, text[previous_end:match.start()]])
                current_key = tuple(match.group().rstrip("\t").split("\t"))
                previous_end = match.end()
            recovered_rows.append([*current_key, text[previous_end:]])
        text_rows = recovered_rows

    texts = pd.DataFrame(text_rows, columns=[*KEYS, "text"])
    joined = labels.merge(texts, on=KEYS, how="outer", validate="one_to_one", indicator=True)
    if not joined["_merge"].eq("both").all():
        raise ValueError(f"{language}/{original_set}: labels and paragraphs do not match")

    label_sets = joined["labels"].map(lambda value: set(filter(None, value.split(","))))
    unknown = set().union(*label_sets) - set(TECHNIQUES)
    if unknown:
        raise ValueError(f"{language}/{original_set}: unknown labels {sorted(unknown)}")
    for technique in TECHNIQUES:
        joined[technique] = label_sets.map(lambda values: int(technique in values))
    joined["paragraph_number"] = joined["paragraph_number"].astype(int)
    joined["original_set"] = original_set
    return joined[["article_id", "paragraph_number", "original_set", "text", *TECHNIQUES]]


semeval = {
    language: pd.concat(
        [load_semeval_split(language, original_set) for original_set in ["train", "dev"]],
        ignore_index=True,
    )
    for language in ["en", "pl"]
}
for language, dataset in semeval.items():
    dataset.to_csv(SEM_EVAL / f"{language}.csv", index=False)
    print(language, dataset.shape, dataset["original_set"].value_counts().to_dict())

en (12625, 27) {'train': 9498, 'dev': 3127}
pl (3110, 27) {'train': 2310, 'dev': 800}
